---
# COSC2753 | Machine Learning

## Task 0: Data Audit and Exploratory Data Analysis
---

# 1. Introduction

This notebook creates the shared evidence and data contract for every model. It audits metadata and images, detects exact duplicates, studies imbalance and target relationships, creates one group-isolated frozen split, and computes training-only RGB statistics. Run and review it before Tasks 1–4.

# 2. Library Imports and Setup

In [ ]:
from pathlib import Path
import hashlib, json, sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'scripts'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import GroupShuffleSplit

from preprocessing import (
    IMAGE_AUDIT_PATH, IMAGE_SIZE, NORMALISATION_PATH, OUTPUT_DIR, SEED, SPLIT_PATH, TARGETS,
    load_metadata, load_prediction_template, preprocess_image,
)
sns.set_theme(style='whitegrid')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 3. Metadata and Coverage Audit

In [ ]:
metadata = load_metadata()
template = load_prediction_template()
coverage_report = {
    'training_rows': len(metadata),
    'training_images_found': int(metadata.has_image.sum()),
    'missing_training_ids': metadata.loc[~metadata.has_image, 'id'].tolist(),
    'test_rows': len(template),
    'missing_test_ids': template.loc[~template.image_path.map(lambda value: Path(value).is_file()), 'id'].tolist(),
    'blank_labels': {target: int(metadata[target].str.strip().eq('').sum()) for target in TARGETS},
}
coverage_report

Literal `NA` usage values are preserved because the shared loader disables automatic missing-value conversion. A genuinely blank label is removed only in that target's modelling notebook. Investigate and document every missing image ID.

# 4. Full Decode and SHA-256 Audit

This pass reads every image and can be slow in a synchronized folder. Make the dataset available offline, set `RUN_FULL_AUDIT=True`, and run it once. The resulting CSV is required by the split section.

In [ ]:
RUN_FULL_AUDIT = False
if RUN_FULL_AUDIT:
    image_rows = []
    for row in metadata.loc[metadata.has_image].itertuples(index=False):
        try:
            digest = hashlib.sha256()
            with Path(row.image_path).open('rb') as handle:
                for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                    digest.update(chunk)
            with Image.open(row.image_path) as image:
                image.load()
                image_rows.append({
                    'id': row.id, 'sha256': digest.hexdigest(), 'width': image.width,
                    'height': image.height, 'mode': image.mode, 'decode_error': '',
                })
        except (OSError, UnidentifiedImageError) as error:
            image_rows.append({
                'id': row.id, 'sha256': '', 'width': '', 'height': '', 'mode': '',
                'decode_error': type(error).__name__,
            })
    image_audit = pd.DataFrame(image_rows)
    image_audit.to_csv(IMAGE_AUDIT_PATH, index=False)
else:
    print('Full audit skipped. Set RUN_FULL_AUDIT=True before creating the final split.')

In [ ]:
if IMAGE_AUDIT_PATH.exists():
    image_audit = pd.read_csv(IMAGE_AUDIT_PATH, dtype={'id': 'string'}, keep_default_na=False)
    hashes = image_audit.loc[image_audit.sha256.ne(''), 'sha256'].value_counts()
    audit_report = {
        **coverage_report,
        'decoded_images': int(image_audit.decode_error.eq('').sum()),
        'decode_errors': image_audit.loc[image_audit.decode_error.ne(''), 'id'].tolist(),
        'exact_duplicate_hash_groups': int(hashes.gt(1).sum()),
        'image_modes': image_audit['mode'].value_counts().to_dict(),
    }
    (OUTPUT_DIR / 'audit.json').write_text(json.dumps(audit_report, indent=2), encoding='utf-8')
    display(audit_report)
else:
    image_audit = None

# 5. Target Distributions and Missing Labels

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
for target, axis in zip(TARGETS, axes.ravel()):
    counts = metadata[target].replace('', '<blank>').value_counts().head(30)
    sns.barplot(x=counts.values, y=counts.index, ax=axis)
    axis.set_title(f'{target}: largest classes')
    axis.set_xlabel('Images')
plt.tight_layout()

In [ ]:
distribution_summary = pd.DataFrame({
    target: {
        'classes': metadata.loc[metadata[target].str.strip().ne(''), target].nunique(),
        'blank': metadata[target].str.strip().eq('').sum(),
        'largest_class_share': metadata[target].value_counts(normalize=True).iloc[0],
        'classes_below_10': int((metadata[target].value_counts() < 10).sum()),
    } for target in TARGETS
}).T
distribution_summary

## 5.1 Distribution observations

Discuss dominant and rare labels for every target. This evidence motivates macro F1 as the main classifier metric. For article type, define head/medium/tail support. For usage, distinguish one genuinely blank value from literal `NA`.

# 6. Target Relationships and Confounding

In [ ]:
pd.crosstab(metadata.articleType, metadata.season, normalize='index').head(25).style.background_gradient(cmap='Blues')

In [ ]:
pd.crosstab(metadata.gender, metadata.usage, normalize='index').style.background_gradient(cmap='Purples')

Explain where season, usage, or gender may be predicted through article-type shortcuts rather than direct visual evidence.

# 7. Visual Inspection

In [ ]:
sample = metadata.loc[metadata.has_image].sample(16, random_state=SEED)
fig, axes = plt.subplots(4, 4, figsize=(10, 12))
for (_, row), axis in zip(sample.iterrows(), axes.ravel()):
    with Image.open(row.image_path) as image:
        axis.imshow(image.convert('RGB'))
    axis.set_title(f'{row.articleType} | {row.season}', fontsize=8)
    axis.axis('off')
plt.tight_layout()

In [ ]:
if image_audit is not None:
    display(image_audit[['width', 'height']].describe())
    display(image_audit['mode'].value_counts())

# 8. Duplicate-Safe Frozen Split

The following code joins rows that share a normalized product display name or exact SHA-256 hash, then assigns whole connected groups to deterministic 70/15/15 splits.

In [ ]:
def build_group_keys(frame, audit):
    frame = frame.merge(audit[['id', 'sha256', 'decode_error']], on='id', how='left')
    frame = frame.loc[frame.has_image & frame.decode_error.eq('')].reset_index(drop=True)
    frame['name_key'] = frame.productDisplayName.astype('string').str.lower().str.replace(r'\W+', ' ', regex=True).str.strip()
    frame.loc[frame.name_key.eq(''), 'name_key'] = frame.id
    parent = list(range(len(frame)))
    def find(index):
        while parent[index] != index:
            parent[index] = parent[parent[index]]; index = parent[index]
        return index
    def union(left, right):
        left_root, right_root = find(left), find(right)
        if left_root != right_root: parent[right_root] = left_root
    for column in ('name_key', 'sha256'):
        values = frame[column].astype('string').fillna('')
        for value, indices in values.groupby(values, sort=False).groups.items():
            if not value or len(indices) < 2: continue
            first = int(indices[0])
            for index in indices[1:]: union(first, int(index))
    frame['group_key'] = [f'group_{find(index)}' for index in range(len(frame))]
    return frame

def create_frozen_split(frame, audit):
    grouped = build_group_keys(frame, audit)
    first = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=SEED)
    _, holdout_index = next(first.split(grouped, groups=grouped.group_key))
    holdout = grouped.iloc[holdout_index]
    second = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=SEED + 1)
    _, test_relative = next(second.split(holdout, groups=holdout.group_key))
    grouped['split'] = 'train'
    grouped.loc[grouped.id.isin(holdout.id), 'split'] = 'validation'
    grouped.loc[grouped.id.isin(holdout.iloc[test_relative].id), 'split'] = 'test'
    return grouped[['id', 'group_key', 'split']]

if SPLIT_PATH.exists():
    splits = pd.read_csv(SPLIT_PATH, dtype={'id': 'string'})
elif image_audit is None:
    raise RuntimeError('Complete the full image audit before creating splits')
else:
    splits = create_frozen_split(metadata, image_audit)
    splits.to_csv(SPLIT_PATH, index=False)
splits['split'].value_counts(), splits.groupby('group_key')['split'].nunique().max()

In [ ]:
split_metadata = metadata.merge(splits, on='id', validate='one_to_one')
pd.concat(
    {target: pd.crosstab(split_metadata[target], split_metadata.split, normalize='columns') for target in TARGETS},
    names=['target', 'label'],
).head(40)

Review split sizes, label proportions, unsupported classes, and confirm that no `group_key` crosses splits. Once modelling begins, do not regenerate this file.

# 9. Training-Only RGB Normalization

In [ ]:
def compute_training_normalisation(frame, split_frame):
    joined = frame.merge(split_frame, on='id', validate='one_to_one')
    paths = joined.loc[joined.split.eq('train'), 'image_path']
    channel_sum = np.zeros(3, dtype=np.float64)
    channel_square_sum = np.zeros(3, dtype=np.float64)
    pixel_count = 0
    for path in paths:
        with Image.open(path) as image:
            array = np.transpose(preprocess_image(image), (1, 2, 0)).astype(np.float64)
        channel_sum += array.sum(axis=(0, 1))
        channel_square_sum += np.square(array).sum(axis=(0, 1))
        pixel_count += array.shape[0] * array.shape[1]
    mean = channel_sum / pixel_count
    std = np.sqrt(channel_square_sum / pixel_count - np.square(mean))
    return {'mean': mean.tolist(), 'std': std.tolist(), 'image_size': list(IMAGE_SIZE)}

if NORMALISATION_PATH.exists():
    with NORMALISATION_PATH.open(encoding='utf-8') as handle:
        normalisation = json.load(handle)
else:
    normalisation = compute_training_normalisation(metadata, splits)
    NORMALISATION_PATH.write_text(json.dumps(normalisation, indent=2), encoding='utf-8')
normalisation

# 10. Decisions and Limitations

Replace this prompt with evidence-backed decisions: RGB conversion, 128×96 resize, exact/name group isolation, 70/15/15 split, training-only normalization and augmentation, macro-F1 selection, blank-label handling, literal-`NA` policy, and limits of visually inferring season, usage, or gender.